In [8]:
1. Dataset Description

Dataset source:
WISDM (Wireless Sensor Data Mining) Dataset http://www.cis.fordham.edu/wisdm/dataset.php

Type of sensor:
Smartphone accelerometer (3-axis acceleration data: X, Y, Z)

Collected variables:

UserID: Identifier of the subject wearing the smartphone

ActivityLabel: Activity performed (e.g., Walking, Jogging, Sitting, Standing, Upstairs, Downstairs)

Timestamp: Time of each sensor reading (milliseconds since epoch)

AccX, AccY, AccZ: Acceleration values along the three axes

Number of samples:
About 1,098,207 sensor readings in the raw dataset (after cleaning, fewer rows)

In [ ]:
import pandas as pd
import time

file_path = r'C:\Users\maham\Downloads\Assignment 5.1\WISDM_ar_v1.1_raw.txt'

def load_wisdm_data(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('user'):
                continue
            parts = line.split(',')
            if len(parts) < 6:  # We expect 6 parts: user, activity, timestamp, x, y, z
                continue
            user, activity, timestamp, x, y, z = parts[:6]
            try:
                data.append({
                    'User': int(user),
                    'Activity': activity,
                    'Timestamp': int(timestamp),
                    'AccX': float(x),
                    'AccY': float(y),
                    'AccZ': float(z)
                })
            except:
                # Skip invalid lines
                continue
    return pd.DataFrame(data)

In [5]:
df = load_wisdm_data(file_path)

print("Data loaded. Shape:", df.shape)
print(df.head())

Data loaded. Shape: (11733, 6)
   User Activity        Timestamp  AccX   AccY  AccZ
0    21  Walking  117003963904000  0.65   9.51  5.13
1    21  Walking  117004003516000  1.73  12.11  7.65
2    21  Walking  117004041541000  1.42  11.18  8.43
3    21  Walking  117004121619000 -3.60   9.77  2.68
4    21  Walking  117004161627000 -2.18   4.40  2.14


In [9]:
current_ms = int(time.time() * 1000)
print("Current time in ms:", current_ms)

Current time in ms: 1764859510580


In [10]:
# Step 2: Filter out invalid timestamps 
df_clean = df[(df['Timestamp'] >= 0) & (df['Timestamp'] <= current_ms)].copy()
print(f"Removed {len(df) - len(df_clean)} rows with invalid timestamps")

# Step 3: Convert 'Timestamp' to datetime with error coercion (invalid become NaT)
df_clean['Timestamp'] = pd.to_datetime(df_clean['Timestamp'], unit='ms', errors='coerce')

# Step 4: Drop rows with NaT timestamps (if any)
num_invalid = df_clean['Timestamp'].isnull().sum()
if num_invalid > 0:
    print(f"Dropping {num_invalid} rows with unconvertible timestamps")
    df_clean = df_clean.dropna(subset=['Timestamp'])

# Step 5: Reset index after filtering
df_clean = df_clean.reset_index(drop=True)

# Now df_clean is safe to use with datetime timestamps
print(df_clean.info())
print(df_clean.head())

Removed 11733 rows with invalid timestamps
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   User       0 non-null      int64         
 1   Activity   0 non-null      object        
 2   Timestamp  0 non-null      datetime64[ns]
 3   AccX       0 non-null      float64       
 4   AccY       0 non-null      float64       
 5   AccZ       0 non-null      float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 132.0+ bytes
None
Empty DataFrame
Columns: [User, Activity, Timestamp, AccX, AccY, AccZ]
Index: []


In [11]:
df_clean = df_clean.sort_values('Timestamp').reset_index(drop=True)


In [12]:
missing_count = df_clean.isnull().sum().sum()
print(f"Total missing values in dataset: {missing_count}")

if missing_count > 0:
    df_clean.fillna(method='ffill', inplace=True)

Total missing values in dataset: 0


In [13]:
df_clean.drop_duplicates(inplace=True)
df_clean.rename(columns={'User': 'UserID', 'Activity': 'ActivityLabel'}, inplace=True)

In [14]:
print(df_clean.info())
print(df_clean.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   UserID         0 non-null      int64         
 1   ActivityLabel  0 non-null      object        
 2   Timestamp      0 non-null      datetime64[ns]
 3   AccX           0 non-null      float64       
 4   AccY           0 non-null      float64       
 5   AccZ           0 non-null      float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 132.0+ bytes
None
Empty DataFrame
Columns: [UserID, ActivityLabel, Timestamp, AccX, AccY, AccZ]
Index: []


In [ ]:
Parse timestamps: Converts raw millisecond timestamps into Python datetime objects, enabling accurate time-based operations.

Filter invalid times: Removes negative or future timestamps that are not logically possible ensuring data validity.

Sort chronologically: Arranges data in proper temporal order, which is crucial for time dependent analysis and modeling.

Handle missing values: Fills gaps in sensor readings to prevent calculation errors and maintain data continuity.

Remove duplicates: Eliminates redundant rows that could bias analysis or machine learning results.

Rename columns: Clarifies the dataset by using clear, descriptive column names, improving readability and ease of use.